# DMRC Contract Intelligence — Retrieval Query Benchmark
## Notebook 04 · Clause & BOQ Benchmark Queries (Direct vs Hybrid)

**Purpose** — A single, re-runnable benchmark of the retrieval layer against representative contract questions, organized as:

- **Section A** — Clause Direct Retrieval (25 queries)
- **Section B** — Clause Hybrid Retrieval (same 25 queries; BM25 / Dense / Hybrid / Reranked shown per query)
- **Section C** — BOQ Direct Retrieval (12 queries)
- **Section D** — BOQ Hybrid Retrieval (same 12 queries)
- **Image & metadata validation** for every recorded top hit
- **Final report** — accuracy tables, Direct-vs-Hybrid comparison, failure cases, suggested improvements

**Corpus note.** The ingested corpus is the **BE-12/BE-14 Lot 3 ECS scope-of-work chapters (63 clause chunks)** plus a **CE-10/CE-11 Lot 4 electrical BOQ (290 chunks: switchboards, ACBs, chiller/pump feeders, cables, meters)**. The benchmark deliberately mixes (a) queries the corpus can answer and (b) classic contract-domain queries (*performance security, arbitration, earthwork, track…*) that this corpus **does not contain** — the correct system behavior for (b) is a low-confidence flag, not a confident wrong answer, and the report scores exactly that.

**Prerequisite** — CPU runtime is sufficient. Run the three setup cells below once per session; every section is independently executable afterwards.

## Setup — repository, dependencies, helpers

In [ ]:
import os
os.chdir("/content" if os.path.isdir("/content") else os.getcwd())
!test -d dmrc_deploy && (echo "dmrc_deploy/ already present -- pulling latest" && cd dmrc_deploy && git pull) \
    || git clone https://github.com/sunvantaconsultancysolutions-design/dmrc_deploy
os.chdir("dmrc_deploy")
!grep -v -E "^(bitsandbytes|nvidia-nvjitlink-cu13)" requirements.txt > /tmp/requirements_retrieval.txt
%pip install -q -r /tmp/requirements_retrieval.txt

In [ ]:

# ============================================================================
# Shared retrieval-validation helpers (self-contained; safe to re-run).
#
# Everything here is a thin display/validation layer over the repository's
# own modules -- no retrieval logic is duplicated. Lazy loading means any
# section of this notebook can be run independently after Sections 1-2
# (clone + install) have been executed once in the session.
# ============================================================================
import os, sys, re, time, json

# Resolve the repository root the same way for Colab (/content) and local runs.
for _cand in ("/content/dmrc_deploy", os.path.abspath("dmrc_deploy"), os.path.abspath(".")):
    if os.path.isfile(os.path.join(_cand, "src", "query.py")):
        REPO = _cand
        break
else:
    raise RuntimeError("dmrc_deploy repository not found -- run the Repository Setup section first.")
os.chdir(REPO)                      # storage.py resolves CHROMA_PATH against CWD
if REPO not in sys.path:
    sys.path.insert(0, REPO)

from IPython.display import display, Markdown, Image as IPyImage

# --- Repository modules (imported, never duplicated) ------------------------
from src import query as q                       # dense search + exact fast paths
from src.hybrid_retriever import hybrid_search   # dense + BM25 + normalized fusion
from src.bm25_index import get_bm25_index        # in-memory BM25 over the corpus
from src.reranker import rerank, evaluate_confidence, expand_with_siblings
from src.prompt_engineering import (
    get_document_name, get_scanned_page, get_boq_item_number, get_boq_page_number,
)
from src.storage import get_collection, COLLECTION_NAME

# NOTE: src.retrieval_caps is deliberately NOT imported in this notebook.
# It monkey-patches hybrid_search()/rerank() in place (as app.py needs in
# production); for validation we want the *uncapped* functions so full
# candidate lists are observable. app.py's behaviour is still exercised,
# because the caps only truncate list length, never change ranking.

_MODELS_READY = False
def ensure_models():
    """Idempotently load the dense encoder, BM25 index, and reranker."""
    global _MODELS_READY
    if _MODELS_READY:
        return
    t0 = time.time()
    q.get_model()                                   # BAAI/bge-m3 (dense)
    get_bm25_index()                                # builds BM25 over ChromaDB corpus
    from src.reranker import get_reranker_model
    get_reranker_model()                            # BAAI/bge-reranker-v2-m3
    _MODELS_READY = True
    print(f"Models + indexes ready in {time.time()-t0:.1f}s "
          f"(collection: {COLLECTION_NAME}, {get_collection().count()} vectors)")

# --- Page-image resolution (mirrors app.py's Rule 1 / Rule 2 exactly) -------
PAGE_IMAGES_DIR = os.path.join(REPO, "page_images")

def resolve_pdf_page(md):
    """Rule 2: the FILE-LOOKUP key is pdf_page (page_number fallback for BOQ)."""
    p = md.get("pdf_page")
    if p in (None, "") and md.get("chunk_type") == "boq":
        p = md.get("page_number")
    try:
        return int(p)
    except (TypeError, ValueError):
        return None

def page_image_path(md):
    doc_id, p = md.get("document_id"), resolve_pdf_page(md)
    if not doc_id or p is None:
        return None
    path = os.path.join(PAGE_IMAGES_DIR, doc_id, f"p{p:04d}.jpg")
    return path if os.path.isfile(path) else None

def show_page_image(md, caption="", width=520):
    path = page_image_path(md)
    if path:
        display(Markdown(f"**{caption}** &nbsp; `{os.path.relpath(path, REPO)}`"))
        display(IPyImage(filename=path, width=width))
        return True
    display(Markdown(f"**{caption}** — *no rendered page image for "
                     f"(document_id={md.get('document_id')!r}, pdf_page={resolve_pdf_page(md)!r})*"))
    return False

def neighbor_page_images(md, radius=1):
    """Related-page helper (NEW in this notebook -- see validation report):
    returns existing image paths for pdf_page +/- radius within the same
    document_id. The repository itself has no neighboring-page helper; this
    is the recommended implementation, kept metadata-only and read-only."""
    doc_id, p = md.get("document_id"), resolve_pdf_page(md)
    if not doc_id or p is None:
        return []
    out = []
    for delta in range(-radius, radius + 1):
        if delta == 0:
            continue
        path = os.path.join(PAGE_IMAGES_DIR, doc_id, f"p{p+delta:04d}.jpg")
        if os.path.isfile(path):
            out.append((p + delta, path))
    return out

# --- Uniform result accessors ------------------------------------------------
def meta(hit):      return hit.get("metadata") or {}
def clause_no(hit): return meta(hit).get("clause_no") or "-"
def boq_item(hit):  return get_boq_item_number(meta(hit)) or "-"
def doc_name(hit):  return get_document_name(meta(hit)) or meta(hit).get("document_id") or "-"
def scan_page(hit):
    s = get_scanned_page(meta(hit))
    return "-" if s in (None, "") else str(s)
def score_of(hit):
    for k in ("reranker_score", "score", "similarity_score", "bm25_score"):
        if hit.get(k) is not None:
            return round(float(hit[k]), 4)
    return None
def preview(hit, n=170):
    t = (hit.get("document") or "").replace("\n", " ")
    return t[: n] + ("…" if len(t) > n else "")

def hits_table(hits, kind="clause", top=5, title=None):
    """Render a compact markdown table for a hit list."""
    rows = []
    for h in hits[:top]:
        ident = clause_no(h) if kind == "clause" else boq_item(h)
        rows.append(f"| {ident} | {preview(h, 88)} | {doc_name(h)[:38]} | "
                    f"{scan_page(h)} | {resolve_pdf_page(meta(h))} | "
                    f"{h.get('retrieval_source','-')} | {score_of(h)} |")
    head = (f"| {'Clause' if kind=='clause' else 'BOQ item'} | Text | Document | "
            "Stamped pg | PDF pg | Source | Score |\n|---|---|---|---|---|---|---|")
    display(Markdown(((f"**{title}**\n\n") if title else "") + head + "\n" + "\n".join(rows)))

# --- Retrieval wrappers used throughout both notebooks -----------------------
def clause_direct(query, top_k=5):
    """Direct clause retrieval = app.py's fast path when the query names a
    clause number (exact metadata lookup + parent-family expansion),
    otherwise clause-filtered pure dense search. The exact path is
    metadata-only -- no model load needed, exactly as in production."""
    no = q.extract_clause_no(query)
    if no:
        hits = list(q.get_chunk_by_clause_no(no))
        seen = {h["chunk_id"] for h in hits}
        for child in q.get_chunks_by_parent_clause(no):
            if child["chunk_id"] not in seen:
                hits.append(dict(child, score=1.0,
                                 retrieval_source="exact_clause_family_match"))
        if hits:
            return hits, f"exact_clause_match ({no})"
    ensure_models()
    hits = q.search(query, top_k=top_k, metadata_filter={"chunk_type": "clause"})
    for h in hits:
        h.setdefault("retrieval_source", "dense")
    return hits, "dense (clause-filtered)"

def boq_direct(query, top_k=5):
    """Direct BOQ retrieval = app.py's BOQ fast path (exact identifier
    lookup on parent/s_no/item_header_no/section_no), otherwise
    BOQ-filtered pure dense search."""
    no = q.extract_boq_item_no(query)
    if no:
        hits = q.get_chunk_by_boq_item_no(no)
        if hits:
            return hits, f"exact_boq_item_match ({no})"
    ensure_models()
    hits = q.search(query, top_k=top_k, metadata_filter={"chunk_type": "boq"})
    for h in hits:
        h.setdefault("retrieval_source", "dense")
    return hits, "dense (boq-filtered)"

def hybrid_pipeline(query, chunk_type=None, top_n=5, expand=False):
    """Full hybrid stack: BM25 + dense -> normalized fusion -> cross-encoder
    rerank (-> optional sibling expansion), with per-stage outputs returned
    so each stage can be displayed and compared."""
    ensure_models()
    flt = {"chunk_type": chunk_type} if chunk_type else None
    bm25 = get_bm25_index().search(query, top_k=10, metadata_filter=flt)
    dense = q.search(query, top_k=10, metadata_filter=flt)
    fused = hybrid_search(query, top_k_dense=30, top_k_bm25=30,
                          final_top_k=60, metadata_filter=flt)
    reranked = rerank(query, fused, top_n=top_n)
    conf = evaluate_confidence(reranked)
    if expand:
        reranked = expand_with_siblings(query, reranked)
    return {"bm25": bm25, "dense": dense, "hybrid": fused,
            "reranked": reranked, "confidence": conf,
            "filter": flt}  # Task 4: expose the filter applied for display

print("Helpers loaded. Repository root:", REPO)


In [ ]:
# ---------------------------------------------------------------------------
# Benchmark bookkeeping + display helpers (thin layer over the shared helpers).
# BENCH_LOG accumulates one record per (section, query) for the final report.
# ---------------------------------------------------------------------------
ensure_models()
BENCH_LOG = []

def _expect_hit(hits, expect):
    """True if any expected keyword/clause label matches within top-3."""
    if not expect:
        return None                     # out-of-corpus probe: no gold answer
    for h in hits[:3]:
        blob = ((h.get("document") or "") + " " + json.dumps(meta(h))).lower()
        if any(e.lower() in blob for e in expect):
            return True
    return False

def record(section, query_text, hits, expect, confidence=None, path="", routing_intent=None):
    top = hits[0] if hits else None
    m = meta(top) if top else {}
    pdf_pg = resolve_pdf_page(m) if top else None
    # Task 2: check neighbor images (prev / next pdf_page)
    neighbor_ok = False
    if top and pdf_pg is not None:
        doc_id_r = m.get("document_id")
        neighbor_ok = any(
            page_image_path({"document_id": doc_id_r, "pdf_page": pdf_pg + d, "chunk_type": m.get("chunk_type")})
            for d in (-1, +1)
        )
    BENCH_LOG.append({
        "section": section, "query": query_text, "path": path,
        "n_hits": len(hits),
        "top_ident": (clause_no(top) if section.startswith("A") or section.startswith("B")
                      else boq_item(top)) if top else None,
        "top_doc": doc_name(top) if top else None,
        "stamped_page": scan_page(top) if top else None,
        "pdf_page": pdf_pg,
        "image_ok": bool(page_image_path(m)) if top else False,
        "neighbor_ok": neighbor_ok,                          # Task 2
        "meta_ok": bool(top and m.get("document_id") and pdf_pg is not None),
        "score": score_of(top) if top else None,
        "confidence": confidence,
        "expected": bool(expect),
        "hit": _expect_hit(hits, expect),
        "routing_intent": routing_intent,                    # Task 4
    })

def show_direct(section, query_text, hits, path, expect, kind):
    ident = clause_no(hits[0]) if kind == "clause" else boq_item(hits[0])
    display(Markdown(f"---\n### ❓ {query_text}\n"
                     f"- **Retrieval path:** `{path}`\n"
                     f"- **{'Clause number' if kind=='clause' else 'BOQ item'}:** `{ident}`\n"
                     f"- **Document:** {doc_name(hits[0])}\n"
                     f"- **Page:** stamped `{scan_page(hits[0])}` · pdf_page `{resolve_pdf_page(meta(hits[0]))}`\n"
                     f"- **Confidence score:** {score_of(hits[0])}"))
    if kind == "boq":
        m = meta(hits[0])
        display(Markdown(f"- **Description:** {preview(hits[0], 200)}\n"
                         f"- **Unit:** `{m.get('unit') or '—'}` · **Quantity:** `{m.get('quantities') or '—'}`"))
    else:
        display(Markdown(f"> {preview(hits[0], 260)}"))
    show_page_image(meta(hits[0]), caption="Exact PDF page")

def show_hybrid(section, query_text, out, kind):
    def one(name, hits):
        if not hits: return f"| {name} | — | — | — | — |"
        h = hits[0]
        ident = clause_no(h) if kind == "clause" else boq_item(h)
        return (f"| {name} | {ident} | {preview(h, 70)} | "
                f"{resolve_pdf_page(meta(h))} | {score_of(h)} |")
    filt = out.get("filter")
    filter_note = (f"  ·  chunk_type filter: `{filt}`"
                   if filt else "  ·  unfiltered pool (general intent)")
    display(Markdown(f"---\n### ❓ {query_text}\n"
            "| Stage | Ident | Top text | pdf_pg | Score |\n|---|---|---|---|---|\n"
            + "\n".join([one("BM25", out["bm25"]), one("Dense", out["dense"]),
                          one("Hybrid", out["hybrid"]), one("Reranked", out["reranked"])])
            + f"\n\n**Confidence gate:** `{out['confidence']}`{filter_note}"))
    if out["reranked"] and out["confidence"]["confident"]:
        show_page_image(meta(out["reranked"][0]), caption="Exact PDF page (reranked top-1)")
    elif not out["confidence"]["confident"]:
        display(Markdown("*Low-confidence — in production `/ask` returns the fixed "
                         "\u201cnot found in context\u201d answer instead of generating.*"))
print("Benchmark helpers ready.")

## Query sets

Each entry is `(query, expected-evidence keywords)`. `expected=[]` marks a deliberate **out-of-corpus probe** — the pass condition for those is the low-confidence gate firing, not a retrieved answer. Coverage notes are grounded in the actual ECS scope-of-work chapters and electrical BOQ ingested into `dmrc_be12be14_ecs`.

In [ ]:
CLAUSE_QUERIES = [
    # -- identifier queries (exact fast path) --------------------------------
    ("Explain clause 1.1",                                   ["scope and purpose"]),
    ("Explain clause 1.2.1",                                 ["1.2.1"]),
    ("Summarize clause 6.8 and all its sub-clauses",         ["6.8"]),
    ("What does clause 6.7.2 cover?",                        ["6.7.2"]),
    ("Explain clause 3.2",                                   ["verification", "validation"]),
    # -- corpus-grounded free-text queries -----------------------------------
    ("What is the scope of work?",                           ["scope"]),
    ("What work is included in the services?",               ["services", "work included"]),
    ("What is the scope of the work of supply?",             ["supply"]),
    ("Which documents are considered relevant documents?",   ["relevant documents"]),
    ("What are the ECS contractor's responsibilities?",      ["responsib"]),
    ("What drawings, documents, records and manuals must be submitted?", ["drawings", "manual"]),
    ("What must be delivered within two months of the Notice to Proceed?", ["two months", "notice to proceed"]),
    ("What are the asset identification requirements?",      ["asset identification"]),
    ("Who are the interfacing contractors and agencies?",    ["interfac"]),
    ("What are the interface requirements with the civil contractor?", ["civil"]),
    ("What are the installation plan and programme requirements?", ["installation"]),
    ("What method statements are required?",                 ["method statement"]),
    ("What resident staff must the contractor provide?",     ["resident staff"]),
    ("What are the prerequisites for installation?",         ["prerequisit"]),
    ("What test programmes and procedures are required?",    ["test programme", "test procedures", "testing"]),
    ("What training must the contractor provide to employer staff?", ["training"]),
    ("What spare parts, tools and test equipment must be provided?", ["spare"]),
    ("What are the operation and maintenance requirements?", ["maintenance"]),
    # -- classic contract-domain probes: NOT in this ECS scope corpus --------
    ("What is the performance security requirement?",        []),
    ("What is the defect liability period?",                 []),
]
assert len(CLAUSE_QUERIES) == 25

BOQ_QUERIES = [
    # -- corpus-grounded (electrical ECS BOQ) --------------------------------
    ("Describe BOQ item 1.02.E.2",                           ["1.02.E.2"]),
    ("cooling tower",                                        ["cooling tower"]),
    ("2500A air circuit breaker",                            ["2500"]),
    ("copper busbar rating",                                 ["busbar"]),
    ("feeder for chiller motors",                            ["chiller"]),
    ("star delta starter for chilled water pumps",           ["star", "pump"]),
    ("current transformers for metering",                    ["metering"]),
    ("XLPE armoured power and control cables",               ["xlpe"]),
    ("digital energy meter",                                 ["energy meter"]),
    ("MCCB with electronic trip unit",                       ["mccb"]),
    # -- classic civil-BOQ probes: NOT in this electrical BOQ ----------------
    ("earthwork excavation quantities",                      []),
    ("track and drainage works",                             []),
]
print(f"{len(CLAUSE_QUERIES)} clause queries, {len(BOQ_QUERIES)} BOQ queries "
      f"({sum(1 for _,e in CLAUSE_QUERIES if not e) + sum(1 for _,e in BOQ_QUERIES if not e)} out-of-corpus probes).")

## Section A — Clause Direct Retrieval

**For every query:** Question · Retrieved clause text · Clause number · Document name · Page number (stamped + pdf) · Confidence score · Exact PDF page image.

*Direct* means the production fast path: exact clause-number metadata lookup (+ parent-family expansion) when the query names a clause, otherwise clause-filtered pure dense search — hybrid fusion and reranking are exercised in Section B.

In [ ]:
for query_text, expect in CLAUSE_QUERIES:
    hits, path = clause_direct(query_text)
    if hits:
        show_direct("A", query_text, hits, path, expect, "clause")
    else:
        display(Markdown(f"---\n### ❓ {query_text}\n*No clause retrieved.*"))
    record("A_clause_direct", query_text, hits, expect, path=path)
print("Section A complete.")

## Section B — Clause Hybrid Retrieval

**Same 25 queries.** For each: BM25 top-1 · Dense top-1 · Hybrid-fused top-1 · Reranked top-1 · confidence-gate verdict · exact page image of the reranked winner. Where the gate reports *not confident*, the production `/ask` endpoint would short-circuit to the fixed no-context answer — that is the **correct** outcome for the out-of-corpus probes.

In [ ]:
for query_text, expect in CLAUSE_QUERIES:
    out = hybrid_pipeline(query_text, chunk_type="clause")
    show_hybrid("B", query_text, out, "clause")
    record("B_clause_hybrid", query_text, out["reranked"], expect,
           confidence=out["confidence"], path="hybrid+rerank")
print("Section B complete.")

## Section C — BOQ Direct Retrieval

**For every query:** BOQ item number · Description · Unit · Quantity · Page · exact PDF page image. Identifier queries take the exact BOQ fast path (`parent`/`s_no`/`item_header_no`/`section_no` metadata match); the rest use BOQ-filtered dense search.

In [ ]:
for query_text, expect in BOQ_QUERIES:
    hits, path = boq_direct(query_text)
    if hits:
        show_direct("C", query_text, hits, path, expect, "boq")
    else:
        display(Markdown(f"---\n### ❓ {query_text}\n*No BOQ item retrieved.*"))
    record("C_boq_direct", query_text, hits, expect, path=path)
print("Section C complete.")

## Section D — BOQ Hybrid Retrieval

**Same BOQ queries** through keyword (BM25) · dense · hybrid · reranked stages, with confidence and the winner's page image.

In [ ]:
for query_text, expect in BOQ_QUERIES:
    out = hybrid_pipeline(query_text, chunk_type="boq")
    show_hybrid("D", query_text, out, "boq")
    record("D_boq_hybrid", query_text, out["reranked"], expect,
           confidence=out["confidence"], path="hybrid+rerank")
print("Section D complete.")

## Image & Metadata Validation

For **every** recorded top hit, verify: correct document resolution · correct page (Rule 2 `pdf_page` present) · page image exists on disk · metadata completeness · related pages available (± 1 neighbor renders exist).

In [ ]:
rows = ["| Sec | Query | Doc ✓ | Page ✓ | Image ✓ | Neighbor ✓ | Meta ✓ | Routing |",
        "|---|---|---|---|---|---|---|---|"]
for r in BENCH_LOG:
    if not r["n_hits"]:
        continue
    img_note = "✅" if r["image_ok"] else ("⚠️ migrate" if "BOQ" in (r.get("top_doc") or "") else "—")
    rows.append(f"| {r['section'].split('_')[0]} | {r['query'][:42]} | "
                f"{'✅' if r['top_doc'] else '❌'} | "
                f"{'✅' if r['pdf_page'] is not None else '❌'} | "
                f"{img_note} | "
                f"{'✅' if r.get('neighbor_ok') else '—'} | "
                f"{'✅' if r['meta_ok'] else '❌'} | "
                f"{r.get('routing_intent') or '—'} |")
display(Markdown("\n".join(rows)))
print()
print("Image column legend:")
print("  ✅ = image exists on disk and resolves via AVAILABLE_PAGES")
print("  ⚠️ migrate = BOQ chunk where migrate_page_image_dirs.py has not yet been run")
print("  — = no image expected (ADDENDUM chunks, or not applicable)")
print()
print("Neighbor column: ✅ = at least one adjacent page (±1) has a rendered image")
print("Routing column: clause/boq/general = Task 4 query_router intent (applied when no fast-path match)")

## Final Report

Generates the eight required outputs from `BENCH_LOG`: (1) overall retrieval accuracy, (2) Direct-vs-Hybrid comparison, (3) clause accuracy, (4) BOQ accuracy, (5) metadata accuracy, (6) page-image accuracy, (7) failure cases, (8) suggested improvements.

In [ ]:
def acc(recs):
    graded = [r for r in recs if r["expected"]]
    if not graded: return "n/a", 0, 0
    ok = sum(1 for r in graded if r["hit"])
    return f"{ok/len(graded):.0%}", ok, len(graded)

def gate(recs):
    probes = [r for r in recs if not r["expected"] and r["confidence"] is not None]
    if not probes: return "n/a"
    correct = sum(1 for r in probes if not r["confidence"]["confident"])
    return f"{correct}/{len(probes)} probes correctly gated"

secs = {s: [r for r in BENCH_LOG if r["section"] == s] for s in
        ["A_clause_direct", "B_clause_hybrid", "C_boq_direct", "D_boq_hybrid"]}

rows = ["## 1 · Retrieval Accuracy Table", "",
        "| Section | Top-3 accuracy (graded queries) | Low-confidence gate |", "|---|---|---|"]
for s, recs in secs.items():
    a, ok, n = acc(recs)
    rows.append(f"| {s} | {a} ({ok}/{n}) | {gate(recs)} |")

da, *_ = acc(secs["A_clause_direct"] + secs["C_boq_direct"])
ha, *_ = acc(secs["B_clause_hybrid"] + secs["D_boq_hybrid"])
rows += ["", "## 2 · Direct vs Hybrid Comparison", "",
         "| Strategy | Accuracy | Strength | Weakness |", "|---|---|---|---|",
         f"| Direct | {da} | authoritative on exact identifiers (score 1.0, no GPU) | paraphrase recall limited to dense arm |",
         f"| Hybrid+rerank | {ha} | best free-text relevance; BM25 rescues exact terms; gated by confidence | ~10× latency of direct; identifier queries shouldn't reach it |"]

ca, cok, cn = acc(secs["A_clause_direct"] + secs["B_clause_hybrid"])
ba, bok, bn = acc(secs["C_boq_direct"] + secs["D_boq_hybrid"])
rows += ["", f"## 3 · Clause Retrieval Accuracy — **{ca}** ({cok}/{cn} graded clause queries)",
         "",  f"## 4 · BOQ Retrieval Accuracy — **{ba}** ({bok}/{bn} graded BOQ queries)"]

with_hits = [r for r in BENCH_LOG if r["n_hits"]]
ma = sum(1 for r in with_hits if r["meta_ok"]) / max(len(with_hits), 1)
ia = sum(1 for r in with_hits if r["image_ok"]) / max(len(with_hits), 1)
rows += ["", f"## 5 · Metadata Accuracy — **{ma:.0%}** of retrieved top hits carry document_id + resolvable pdf_page",
         "",  f"## 6 · Page Image Accuracy — **{ia:.0%}** of retrieved top hits resolve to an existing page render",
         "", "## 7 · Failure Cases", "",
         "| Section | Query | Issue |", "|---|---|---|"]
fails = 0
for r in BENCH_LOG:
    if r["expected"] and r["hit"] is False:
        rows.append(f"| {r['section']} | {r['query'][:50]} | expected evidence not in top-3 |"); fails += 1
    if not r["expected"] and r["confidence"] is not None and r["confidence"]["confident"]:
        rows.append(f"| {r['section']} | {r['query'][:50]} | out-of-corpus probe passed the confidence gate |"); fails += 1
    if r["n_hits"] and not r["image_ok"]:
        rows.append(f"| {r['section']} | {r['query'][:50]} | top hit has no rendered page image |"); fails += 1
if not fails:
    rows.append("| — | — | no failures recorded |")
display(Markdown("\n".join(rows)))

## 8 · Fix Status & Remaining Action Items

The issues identified when this benchmark was written have been addressed in `dmrc_deploy_fixes.zip`.

| # | Issue | Status |
|---|---|---|
| 1 | Route identifier queries away from hybrid | ✅ Already done in `/ask` (unchanged) |
| 2 | Add neighboring-page URLs to `/ask` sources | ✅ **FIXED** — `prev_image_url` / `next_image_url` in `SourceItem` (Task 2) |
| 3 | Activate figure retrieval (`manifest.json`) | ✅ **DOCUMENTED** — 0 figures extractable from scanned PDFs; activates on new embedded-diagram content |
| 4 | Fix BOQ image linkage (slug mismatch) | ✅ **FIXED** — `render_pages.py` derives keys via `_slugify`; `migrate_page_image_dirs.py` renames dirs (Task 1) |
| 5 | Query-intent routing for free-text | ✅ **FIXED** — `src/query_router.py` + filter passed to `hybrid_search()` (Task 4) |
| 6 | Recalibrate confidence gate | 📌 Run `scripts/calibrate_confidence.py` after corpus changes |
| 7 | Grow graded query sets | 📌 Add one line per query to `CLAUSE_QUERIES` / `BOQ_QUERIES` |

**ADDENDUM image linkage (211 chunks):** still unresolved — requires manual verification of the pdf_page offset within the Vol-3-38-47 PDF binding. See CHANGELOG Known Limitation 1.

**Deployment steps for this benchmark to show ✅ on all BOQ image rows:**
```bash
python scripts/migrate_page_image_dirs.py   # rename existing page_images dirs
# restart server, then re-run this benchmark
```